# Final Results Summary

Run this after notebooks 02 and 03. It loads the saved training and analysis outputs, builds final report tables/figures, and backs them up to Google Drive when running in Colab.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_packages() -> None:
    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "umap": "umap-learn",
        "torch": "torch",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
        "huggingface_hub": "huggingface_hub",
    }
    missing = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required analysis packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "FinalProject"]:
        candidate = candidate.resolve()
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root locally.")



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_packages()

if IS_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

def module_uses_code_root(module_file: str | None, code_root: Path) -> bool:
    if module_file is None:
        return False
    try:
        Path(module_file).resolve().relative_to(code_root.resolve())
        return True
    except ValueError:
        return False


os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

src_module = sys.modules.get("src")
src_file = getattr(src_module, "__file__", None)
if src_module is not None and not module_uses_code_root(src_file, CODE_ROOT):
    stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

In [ ]:
from pathlib import Path

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_report_dir = PROJECT_ROOT / 'artifacts' / 'reports' / experiment_name
local_report_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root is not None else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root is not None else None
drive_report_dir = drive_run_root / 'final_report' if drive_run_root is not None else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir is not None and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir

analysis_dir = local_analysis_dir
if not (analysis_dir / 'steering_summary.csv').exists() and drive_analysis_dir is not None and (drive_analysis_dir / 'steering_summary.csv').exists():
    analysis_dir = drive_analysis_dir

required_checkpoint_files = [
    checkpoint_dir / 'model_state.pt',
    checkpoint_dir / 'val_metrics.json',
    checkpoint_dir / 'test_metrics.json',
    checkpoint_dir / 'test_predictions.csv',
    checkpoint_dir / 'label_mapping.json',
]
required_analysis_files = [
    analysis_dir / 'layerwise_centroid_metrics.csv',
    analysis_dir / 'centroid_cosine_matrix.csv',
    analysis_dir / 'projection_summary_test_centered.csv',
    analysis_dir / 'steering_summary.csv',
]
missing_files = [str(path) for path in required_checkpoint_files + required_analysis_files if not path.exists()]
assert not missing_files, 'Missing final-report inputs: ' + ', '.join(missing_files)

print('Checkpoint source:', checkpoint_dir)
print('Analysis source:', analysis_dir)
print('Local report dir:', local_report_dir)
if drive_report_dir is not None:
    print('Drive report dir:', drive_report_dir)


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from src.analysis.final_report import (
    build_best_layer_summary,
    build_confusion_matrix_frame,
    build_overall_metrics_frame,
    build_projection_alignment_frame,
    build_steering_summary_frame,
    build_takeaways_markdown,
    load_final_artifacts,
)

artifacts = load_final_artifacts(checkpoint_dir=checkpoint_dir, analysis_dir=analysis_dir)
overall_metrics_df = build_overall_metrics_frame(artifacts)
confusion_df = build_confusion_matrix_frame(artifacts)
best_layer_summary = build_best_layer_summary(artifacts)
projection_alignment_df = build_projection_alignment_frame(artifacts)
steering_selected_df = build_steering_summary_frame(artifacts)
summary_markdown = build_takeaways_markdown(artifacts)

(overall_metrics_df.round(4)).to_csv(local_report_dir / 'overall_metrics.csv', index=False)
confusion_df.to_csv(local_report_dir / 'test_confusion_matrix.csv')
projection_alignment_df.to_csv(local_report_dir / 'projection_alignment_summary.csv', index=False)
steering_selected_df.to_csv(local_report_dir / 'steering_selected_alpha_summary.csv', index=False)
(local_report_dir / 'final_results_summary.md').write_text(summary_markdown, encoding='utf-8')

print('Best layer summary:', best_layer_summary)
display(overall_metrics_df.round(4))
display(confusion_df)
display(projection_alignment_df)
display(steering_selected_df.round(4))
display(Markdown(summary_markdown))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 6))
sns.heatmap(confusion_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Test Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig(local_report_dir / 'test_confusion_matrix.png', dpi=200)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

steering_df = artifacts.steering_summary.copy()
steering_df['alpha'] = steering_df['alpha'].astype(float)

plt.figure(figsize=(9, 5))
sns.lineplot(
    data=steering_df,
    x='alpha',
    y='delta_target_prob_all',
    hue='target_label',
    marker='o',
)
plt.title('Steering Effect by Direction and Alpha')
plt.xlabel('Alpha')
plt.ylabel('Delta target probability (all test samples)')
plt.tight_layout()
plt.savefig(local_report_dir / 'steering_delta_curves.png', dpi=200)
plt.show()


In [ ]:
test_predictions = artifacts.test_predictions.copy()
misclassified = test_predictions[test_predictions['true_label'] != test_predictions['pred_label']].copy()
misclassification_summary = (
    misclassified.groupby(['true_label', 'pred_label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
misclassification_summary.to_csv(local_report_dir / 'misclassification_summary.csv', index=False)

print('Total test samples:', len(test_predictions))
print('Misclassified samples:', len(misclassified))
display(misclassification_summary.head(10))


In [ ]:
print('Report directory:', local_report_dir)
for path in sorted(local_report_dir.iterdir()):
    print(path.name)


In [ ]:
if IS_COLAB:
    import shutil

    assert drive_report_dir is not None
    if drive_report_dir.exists():
        shutil.rmtree(drive_report_dir)
    shutil.copytree(local_report_dir, drive_report_dir)

    print('Backed up final report artifacts to Google Drive:')
    print(' -', drive_report_dir)
    print('\nDrive final report contents:')
    for path in sorted(drive_report_dir.iterdir()):
        kind = 'dir' if path.is_dir() else 'file'
        print(f' [{kind}] {path.name}')
else:
    print('Google Drive backup runs only in Colab.')
